# HAI Benchmark — Gradient Boosting vs Deep Learning

UREx CDE2605R — benchmarking XGBoost against LSTM sequence models for anomaly
detection in Industrial Control Systems, run on the HAI dataset.

This notebook is a runnable, annotated version of `src/experiment.py --dataset
hai`. It imports directly from the project's `src/` modules (`config.py`,
`data.py`, `models.py`, `evaluate.py`, `experiment.py`) rather than
re-implementing any logic, so results here are identical to running the CLI.

**Dataset note.** The HAI file used here (`data/raw/hai/hai3_test_220812.csv`)
is a single 15-hour, 1&nbsp;Hz, labelled test-split excerpt ("hai3") sourced
from course material, not the full multi-day HAI 21.03 release referenced in
`notes/datasets.md`. It contains genuine HAI attack labels (14 attacks, 86
sensor/actuator columns) and is enough to sanity-check the pipeline on real
data, but it is not the complete dataset the project ultimately calls for.

**The project's four methodological commitments** (see `README.md`), all
enforced by the code this notebook calls:
1. Every metric is reported under both the strict *pointwise* protocol and the
   inflationary *point-adjust* protocol used by most published ICS papers.
2. Splits are chronological — no shuffling, no `random_state`.
3. Scalers are fit on training data only.
4. Thresholds are chosen on validation, applied unchanged to test.


## Setup

The two environment variables below must be set **before** XGBoost or PyTorch
are imported anywhere in this process. Both libraries bundle their own copy of
the OpenMP runtime (`libomp.dylib`), and on macOS, loading both into one
process without this can deadlock silently — the kernel just hangs with 0%
CPU, no error, no traceback. This was hit and diagnosed during CLI runs of
this same pipeline.


In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

# This notebook lives in notebooks/; the project modules live in ../src.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
assert SRC.exists(), f"expected {SRC} to exist — run this notebook from notebooks/"
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd

print(f"project root: {ROOT}")


project root: /Users/santhosh/Desktop/urex-ics


## Load the HAI dataset

`data.load("hai")` calls `load_hai()` (reads the raw CSV, parses the attack
windows out of the label report, builds the `label` column) and then
`chronological_split()` (60% train / 20% val / 20% test, in time order,
scaler fit on train only). Both steps and their guardrails live in
`src/data.py`.


In [2]:
from config import RESULTS, THRESHOLD_PERCENTILE
from data import load

split = load("hai")
print(split.summary())


hai: 86 features
  train   32,400 rows  anomaly rate 0.0602
  val     10,800 rows  anomaly rate 0.0247
  test    10,800 rows  anomaly rate 0.0706


Note the anomaly rate differs across splits (train 6.0%, val 2.5%, test
7.1% in the run this notebook was authored against) — expected for a
chronological split, since the 14 real attacks in this recording are not
evenly spaced through the 15-hour window. `chronological_split` asserts the
test split contains at least one anomaly; if it didn't, that would mean the
split fractions put every attack in train/val and the result would be
meaningless.


In [3]:
print(f"features: {split.n_features}")
print(f"first 10 feature names: {split.feature_names[:10]}")


features: 86
first 10 feature names: ['P1_FCV01D', 'P1_FCV01Z', 'P1_FCV02D', 'P1_FCV02Z', 'P1_FCV03D', 'P1_FCV03Z', 'P1_FT01', 'P1_FT01Z', 'P1_FT02', 'P1_FT02Z']


## The five detectors

All five share one interface — `detector.fit(split)` then
`detector.score(X)` — defined in `src/models.py`. That uniformity is what
lets the same evaluation code run over every model without special cases.

| name | family | supervised? | what the score measures |
|---|---|---|---|
| `random` | baseline | – | pure noise; the sanity floor every real model must beat |
| `xgboost` | gradient boosting | yes | each row scored independently, no explicit memory of the past |
| `xgboost_lags` | gradient boosting | yes | same, plus engineered lag/rolling/diff features so it can approximate short-term memory |
| `lstm_ae` | deep (autoencoder) | no (trained on normal rows only) | reconstruction error over a 60-second window — measures *compressibility* |
| `lstm_forecast` | deep (forecaster) | no (trained on normal rows only) | one-step prediction error — measures *predictability*, sees temporal structure the autoencoder cannot |

The deep family is deliberately represented by **two** models. Reporting only
the autoencoder would misattribute a scoring-rule limitation to deep learning
as a whole — see `LSTMAutoencoderDetector`'s docstring in `models.py` for the
full argument.


In [4]:
from models import REGISTRY

MODEL_NAMES = ["random", "xgboost", "xgboost_lags", "lstm_ae", "lstm_forecast"]
print("registered models:", list(REGISTRY))


registered models: ['random', 'xgboost', 'xgboost_lags', 'lstm_ae', 'lstm_forecast']


## Evaluation protocol: pointwise vs point-adjust

This is the project's central methodological point (`src/evaluate.py`).

- **Pointwise** — each second judged independently. Strict, and the honest
  number to read first.
- **Point-adjust** — if a model flags *any one second* inside a ground-truth
  attack window, the *entire window* counts as detected. This is the
  dominant protocol in published ICS anomaly detection work, and Kim et al.
  (AAAI 2022) showed it lets a random score beat published state of the art.

`random_baseline_check()` reproduces that result directly: pure noise scored
against synthetic anomaly segments. If point-adjusted F1 comes back high
while pointwise F1 stays near zero, the metric implementation is correct —
and the literature's typical benchmark numbers are exposed as an artefact
of the protocol, not evidence the model actually works.


In [5]:
from evaluate import random_baseline_check

sanity = random_baseline_check()
print(sanity.summary())


random noise on synthetic
  anomaly rate     0.0953   threshold 0.9903
  ROC-AUC          0.4979    PR-AUC 0.0948
  pointwise      P 0.108  R 0.011  F1 0.021  MCC 0.004
  point-adjusted P 0.910  R 0.945  F1 0.927  MCC 0.919
  F1 inflation     +0.906


## Run the benchmark

`experiment.run()` fits each detector on the training window, picks a
threshold at the 97th percentile of *validation* scores (never test), scores
the test window, evaluates under both protocols, prints a summary per model,
and saves the full results as JSON under `results/`.

The two LSTM models are the slow part (8 and 20 training epochs respectively
over ~32k rows) — this cell takes several minutes.


In [6]:
from experiment import run

results = run(split, MODEL_NAMES, tag="notebook")


hai: 86 features
  train   32,400 rows  anomaly rate 0.0602
  val     10,800 rows  anomaly rate 0.0247
  test    10,800 rows  anomaly rate 0.0706

  random ...
random on hai
  anomaly rate     0.0706   threshold 0.9708
  ROC-AUC          0.4927    PR-AUC 0.0710
  pointwise      P 0.073  R 0.030  F1 0.043  MCC 0.002
  point-adjusted P 0.722  R 1.000  F1 0.839  MCC 0.837
  F1 inflation     +0.796

  xgboost ...


xgboost on hai
  anomaly rate     0.0706   threshold 0.3863
  ROC-AUC          0.8967    PR-AUC 0.4475
  pointwise      P 0.520  R 0.558  F1 0.538  MCC 0.502
  point-adjusted P 0.660  R 1.000  F1 0.795  MCC 0.797
  F1 inflation     +0.257

  xgboost+lags[1, 5, 30] ...


xgboost+lags[1, 5, 30] on hai
  anomaly rate     0.0706   threshold 0.4884
  ROC-AUC          0.8530    PR-AUC 0.3148
  pointwise      P 0.382  R 0.172  F1 0.237  MCC 0.220
  point-adjusted P 0.782  R 1.000  F1 0.878  MCC 0.875
  F1 inflation     +0.641

  lstm_ae(w=60,h=64) ...


    epoch   0  loss 0.33431


    epoch   1  loss 0.17386


...

    epoch  16  loss 0.05003


    epoch  17  loss 0.04937


    epoch  18  loss 0.04885


    epoch  19  loss 0.04826


lstm_forecast(w=60,h=64) on hai
  anomaly rate     0.0706   threshold 1.382
  ROC-AUC          0.9802    PR-AUC 0.7819
  pointwise      P 0.864  R 0.915  F1 0.888  MCC 0.880
  point-adjusted P 0.874  R 1.000  F1 0.933  MCC 0.930
  F1 inflation     +0.044

saved -> results/hai-notebook-20260826-184416.json

model                        PR-AUC    F1 pw   F1 adj  inflation
----------------------------------------------------------------
random                        0.071    0.043    0.839     +0.796
xgboost                       0.448    0.538    0.795     +0.257
xgboost+lags[1, 5, 30]        0.315    0.237    0.878     +0.641
lstm_ae(w=60,h=64)            0.801    0.726    0.753     +0.027
lstm_forecast(w=60,h=64)      0.782    0.888    0.933     +0.044

Large inflation means the model is essentially guessing and getting
lucky once per attack window. Read the pointwise column first.


## Results, tidied

Same numbers as the printed table above, reshaped into a DataFrame for
easier reading/plotting. `f1_inflation` is `f1_point_adjust - f1_pointwise`
— the headline number. Large inflation means a model is close to guessing
and getting lucky once per attack window; read the pointwise column first.


In [7]:
summary = pd.DataFrame([
    {
        "model": r.meta.get("model"),
        "roc_auc": r.roc_auc,
        "pr_auc": r.pr_auc,
        "f1_pointwise": r.pointwise.f1,
        "f1_point_adjust": r.point_adjusted.f1,
        "f1_inflation": r.f1_inflation,
    }
    for r in results
]).set_index("model")

summary


,roc_auc,pr_auc,f1_pointwise,f1_point_adjust,f1_inflation
model,,,,,
random,0.492732,0.070998,0.042672,0.838745,0.796074
xgboost,0.896696,0.447514,0.538315,0.795407,0.257092
"xgboost+lags[1, 5, 30]",0.852978,0.314792,0.237104,0.877880,0.640776
"lstm_ae(w=60,h=64)",0.973635,0.800641,0.726263,0.753337,0.027074
"lstm_forecast(w=60,h=64)",0.980187,0.781900,0.888464,0.932681,0.044217


## How to read this

- **`random`'s `f1_inflation`** should be large and positive — that's the
  Kim et al. effect confirmed on real HAI data, not just the synthetic
  sanity check above. If any real model's inflation is comparably large,
  its point-adjusted score is not trustworthy evidence it works.
- **Deep vs boosting**: HAI is registered in `config.py` as
  `temporal_dependency: high`. The project's hypothesis is that sequence
  models (`lstm_ae`, `lstm_forecast`) have a structural advantage here over
  row-independent gradient boosting (`xgboost`) — check whether
  `f1_pointwise` bears that out.
- **`xgboost` vs `xgboost_lags`**: whether engineered lag features close the
  gap with the deep models, widen it, or make boosting worse — all three
  are informative outcomes for the project's central question (how much of
  deep learning's advantage is architectural vs recoverable with feature
  engineering).
- Full precision/recall/MCC per model, per protocol, is in the saved JSON
  under `results/hai-notebook-<timestamp>.json` (path printed above).
